# Baseline: Qwen3-8B-Base on MBPP, MBPP+, LiveCodeBench v5

Frozen reference for every later thesis stage (GoT, GRP-SFT, PASC-GRPO, multi-agent).

| Benchmark | Sample | Completions per task | Metric |
|---|---|---|---|
| MBPP | 100 of 500 (uniform random) | 16 | avg@16 |
| MBPP+ | 100 of 378 (uniform random) | 16 | avg@16 |
| LiveCodeBench v5 | 100 of 880 (stratified by difficulty x execution mode) | 3 | avg@3 |

Steps: download datasets, draw samples, generate with vLLM, grade in a sandbox, summarize, verify.
Helpers live in `baseline_utils.py`. Run this notebook on the Vast GPU server.

In [ ]:
import json, platform, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import baseline_utils as bu

ROOT = Path.cwd().parent                       # repo root (notebook lives in baseline/)
DATASETS_DIR = ROOT / "datasets"
RUN_ID = "qwen3-8b-base_v1"
RUN_DIR = ROOT / "baseline" / "results" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

def log(msg):
    line = f"[{datetime.now():%H:%M:%S}] {msg}"
    print(line, flush=True)
    with open(RUN_DIR / "run.log", "a") as f:
        f.write(line + "\n")

## Locked configuration
Changing anything here means a new `RUN_ID`, never an edit of `qwen3-8b-base_v1`.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B-Base"
MODEL_REVISION = "49e3418fbbbca6ecbdf9608b4d22e5a407081db4"
DATASET_REVISIONS = {
    "mbpp": "4bb6404fdc6cacfda99d4ac4205087b89d32030c",
    "mbpp_plus": "b2d74c91837c3f2a20c1299ae98133cbe7cfa077",
    "livecodebench_v5": "0fe84c3912ea0c4d4a78037083943e8f0c4dd505",
}
SAMPLE_SIZE, SAMPLE_SEED = 100, 42
K = {"mbpp": 16, "mbpp_plus": 16, "livecodebench_v5": 3}       # completions per task
DECODING = {"temperature": 0.6, "top_p": 0.95, "seed": 42}
MAX_COMPLETION_TOKENS = 8192
TIMEOUT_SEC = {"mbpp": 10, "mbpp_plus": 30, "livecodebench_v5": 6}   # LCB: per test case

## 1. Download datasets
Files land in `datasets/<benchmark>/raw/` at the pinned commits. Already-downloaded files are skipped.

In [ ]:
log("downloading datasets")
bu.download_datasets(DATASETS_DIR, DATASET_REVISIONS)
log("datasets ready")

## 2. Draw representative samples
* MBPP, MBPP+: simple random sample (seed 42).
* LiveCodeBench v5: proportional stratified sample over difficulty x execution mode (stdin vs starter code), so the mix of easy/medium/hard and judge types matches the full set.

Task ids are saved to `datasets/samples/` and reused by every later experiment.

In [ ]:
full = {
    "mbpp": bu.load_mbpp(DATASETS_DIR),
    "mbpp_plus": bu.load_mbpp_plus(DATASETS_DIR),
    "livecodebench_v5": bu.load_livecodebench(DATASETS_DIR, keep_tests_for=set()),
}
for name, tasks in full.items():
    assert len(tasks) == bu.DATASETS[name]["expected_tasks"], f"{name}: dataset mismatch ({len(tasks)} tasks)"

STRATA = ("difficulty", "execution_mode")
sample_ids = {
    "mbpp": bu.uniform_sample([t["task_id"] for t in full["mbpp"]], SAMPLE_SIZE, SAMPLE_SEED),
    "mbpp_plus": bu.uniform_sample([t["task_id"] for t in full["mbpp_plus"]], SAMPLE_SIZE, SAMPLE_SEED),
    "livecodebench_v5": bu.stratified_sample(full["livecodebench_v5"], SAMPLE_SIZE, SAMPLE_SEED, STRATA),
}
(DATASETS_DIR / "samples").mkdir(exist_ok=True)
for name, ids in sample_ids.items():
    (DATASETS_DIR / "samples" / f"{name}_task_ids.json").write_text(json.dumps(ids, indent=1) + "\n")

bu.strata_table(full["livecodebench_v5"], sample_ids["livecodebench_v5"], STRATA)

In [ ]:
# Sampled tasks only; LiveCodeBench hidden tests are decoded for these 100 problems.
tasks = {name: [t for t in full[name] if t["task_id"] in set(ids)] for name, ids in sample_ids.items()}
tasks["livecodebench_v5"] = bu.load_livecodebench(DATASETS_DIR, keep_tests_for=set(sample_ids["livecodebench_v5"]))
tasks["livecodebench_v5"] = [t for t in tasks["livecodebench_v5"] if "tests" in t]
{name: len(t) for name, t in tasks.items()}

## 3. Manifest

In [ ]:
import vllm
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
manifest = {
    "run_id": RUN_ID,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "precision": "bfloat16",
    "benchmarks": {b: {"k": K[b], "max_completion_tokens": MAX_COMPLETION_TOKENS, "num_tasks": SAMPLE_SIZE,
                       "sampling": "stratified" if b == "livecodebench_v5" else "uniform",
                       "dataset_repo": bu.DATASETS[b]["repo"], "dataset_revision": DATASET_REVISIONS[b],
                       "grading_timeout_sec": TIMEOUT_SEC[b]} for b in K},
    "sample_seed": SAMPLE_SEED,
    "decoding": {**DECODING, "seed_scheme": "seed + sample_idx per request", "stop": ["</answer>"]},
    "prompt_templates_sha256": bu.template_hashes(),
    "prompt_format": "tokenizer chat template, single user turn with a one-shot format example; "
                     "assistant turn prefilled with response_prefix",
    "response_prefix": bu.RESPONSE_PREFIX,
    "scoring": "primary: code only from <answer>; secondary avg_at_k_fallback also accepts the last fenced code block",
    "reasoning_tokens_rule": "text after response_prefix up to </think>; else up to <answer>; else to the end",
    "hardware": gpu,
    "vllm_version": vllm.__version__,
    "python_version": platform.python_version(),
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
manifest

## 4. Generate
One vLLM engine for all three benchmarks. A benchmark whose `raw_generations.jsonl` already exists is skipped, so a restart reuses saved completions instead of regenerating them.

In [ ]:
from vllm import LLM

llm = LLM(MODEL_ID, revision=MODEL_REVISION, dtype="bfloat16", seed=DECODING["seed"],
          max_model_len=12288, gpu_memory_utilization=0.90, enable_prefix_caching=True,
          disable_log_stats=False)
tokenizer = llm.get_tokenizer()

In [ ]:
for bench in K:
    out = RUN_DIR / bench / "raw_generations.jsonl"
    if out.exists():
        log(f"{bench}: raw generations already saved, skipping")
        continue
    log(f"{bench}: generation started")
    records, wall = bu.generate(llm, tokenizer, tasks[bench], K[bench], DECODING, MAX_COMPLETION_TOKENS, log)
    bu.write_jsonl(out, records)
    (RUN_DIR / bench / "generation_wall_clock_sec.txt").write_text(f"{wall:.1f}\n")
    log(f"{bench}: saved {len(records)} completions")

## 5. Grade
Each completion runs in a fresh Python process as user `nobody` with time and memory limits.
Grading reads only the saved raw generations, so it can be repeated on CPU at any time.

The main score is strict: code is taken only from `<answer>...</answer>`. `avg@k fallback` is a secondary score that also accepts the last fenced code block when the answer tags are missing.

In [ ]:
summaries = {}
for bench in K:
    raw = bu.recount_reasoning_tokens(bu.read_jsonl(RUN_DIR / bench / "raw_generations.jsonl"), tokenizer)
    bu.write_jsonl(RUN_DIR / bench / "raw_generations.jsonl", raw)   # derived token field only
    log(f"{bench}: grading {len(raw)} completions")
    evals = bu.grade_all(raw, tasks[bench], TIMEOUT_SEC[bench])
    bu.write_jsonl(RUN_DIR / bench / "eval_results.jsonl", evals)
    wall = float((RUN_DIR / bench / "generation_wall_clock_sec.txt").read_text())
    summaries[bench] = bu.summarize(RUN_ID, bench, K[bench], raw, evals, wall)
    (RUN_DIR / bench / "summary.json").write_text(json.dumps(summaries[bench], indent=2) + "\n")
    log(f"{bench}: avg@{K[bench]} = {summaries[bench]['metrics'][f'avg_at_{K[bench]}']:.4f}")

## 6. Results
Paper numbers (GRP, Liu et al. 2026, Tables 1 and 3) are full-dataset scores, shown for context only.
Our scores come from 100-task samples, so some gap is expected.

In [ ]:
PAPER = {"mbpp": 0.7327, "mbpp_plus": 0.6173, "livecodebench_v5": 0.2976}
table = pd.DataFrame([
    {"benchmark": b, "metric": f"avg@{K[b]}", "score": s["metrics"][f"avg_at_{K[b]}"],
     "fallback_score": s["metrics"][f"avg_at_{K[b]}_fallback"], "paper_full_set": PAPER[b],
     **{m: s["metrics"][m] for m in ["format_compliance_rate", "syntax_error_rate", "truncation_rate",
                                     "mean_reasoning_tokens", "mean_total_tokens", "tokens_per_correct_solution",
                                     "mean_latency_sec", "tokens_per_sec"]}}
    for b, s in summaries.items()
]).set_index("benchmark")
table.T

In [ ]:
pd.Series(summaries["livecodebench_v5"]["metrics"]["avg_at_3_by_difficulty"], name="LCB v5 avg@3 by difficulty")

## 7. Verify
Acceptance checks: 100 unique task ids per benchmark, exactly 16 / 16 / 3 graded completions per task, no missing or duplicate records, one summary per benchmark. Then a self-comparison that must show zero deltas, and checksums of every artifact (written last).

In [ ]:
display(bu.verify_run(RUN_DIR, K, sample_ids))
self_cmp = subprocess.run([sys.executable, "compare_runs.py", str(RUN_DIR), str(RUN_DIR)],
                          capture_output=True, text=True, check=True).stdout
print(self_cmp)
log("baseline complete and verified")
bu.write_checksums(RUN_DIR)